# GPT5-mini text generation (rewriting Claude 4.5 Haiku answers)

### Import libraries

In [1]:
import numpy as np
import pandas as pd

import os
import json

from google.colab import drive, userdata
from openai import OpenAI

### Set up Google Drive mounting and define input/output paths:

In [2]:
# Mount the google drive folder
drive.mount('/content/drive', force_remount = True) # force reconnection

# Define directory, input, and output paths
project_directory = "/content/drive/MyDrive/Colab Notebooks/DS266/final_project"
input_file = os.path.join(project_directory, "data/everything_but_reddit_unstandardized_with_kimi.parquet")
output_file = os.path.join(project_directory, "data/gpt data/gpt5mini_claude.jsonl")

# Ensure the output directory exists
if not os.path.exists(project_directory):
  os.makedirs(project_directory)

# Get access to the seceret api key
os.environ["OPENAI_API_KEY"] = userdata.get('HC3_GPT5mini')
client = OpenAI()

Mounted at /content/drive


### Read the input file:

In [3]:
# Parquet file
df = pd.read_parquet(input_file)
print('Sucessfully read the input file with the questions and original answers!')

print(f'The input file contains {len(df)} records/rows.')
print(f'Column names: {list(df.columns)}')

print('\nThe first 5 rows:')
display(df.head())

Sucessfully read the input file with the questions and original answers!
The input file contains 5306 records/rows.
Column names: ['question', 'human_answers', 'source', 'gemini_answers', 'gpt5mini_answer', 'claude_haiku_4.5', 'kimi_k2_answers']

The first 5 rows:


,question,human_answers,source,gemini_answers,gpt5mini_answer,claude_haiku_4.5,kimi_k2_answers
0,what composer used sound mass,"['Composers and works include Barbara Kolb , P...",open_qa,The most prominent composer strongly associate...,"Several 20th-century composers used the ""sound...",# Composers Who Used Sound Mass\n\n**Gyorgy Li...,"The term **""sound mass""** (or **""sound-mass co..."
1,where did the persian war take place,['The Greco-Persian Wars (also often called th...,open_qa,The Persian Wars primarily took place in **mai...,If you mean the Greco–Persian Wars (often call...,# The Persian Wars\n\nThe Persian Wars took pl...,The **Persian Wars** (also called the **Greco-...
2,what are add ons,"['Plug-in (computing) , a piece of software wh...",open_qa,An **add-on** is something extra or additional...,"""Add-ons"" (sometimes written ""add ons"", ""addon...",# Add-ons\n\nAdd-ons are **additional features...,**Add-ons** are extra software components or e...
3,how does a dredge work?,['Dredging is an excavation activity or operat...,open_qa,A dredge is essentially a floating machine des...,"A dredge is a machine used to remove sediment,...",# How a Dredge Works\n\nA dredge is a vessel o...,A **dredge** is a machine or system designed t...
4,what classes are considered humanities,['The humanities are academic disciplines that...,open_qa,The humanities are academic disciplines that s...,Humanities are academic disciplines that study...,# Humanities Classes\n\nHumanities typically i...,“Humanities” is the broad family of discipline...


### Print the sample question and LLM answer:

In [4]:
print('Sample question:')
display(df['question'][0])

print('\nSample Claude original answer:')
display(df['claude_haiku_4.5'][0])

Sample question:


'what composer used sound mass'


Sample Claude original answer:


'# Composers Who Used Sound Mass\n\n**Gyorgy Ligeti** is the most prominent composer associated with sound mass technique. He pioneered and popularized this compositional approach, particularly in works like:\n- *Atmosphères* (1961)\n- *Lontano* (1967)\n- *Melodien* (1971)\n\n## Other Notable Composers:\n\n- **Krzysztof Penderecki** - used dense, textural orchestration\n- **Iannis Xenakis** - employed mathematical and granular sound concepts\n- **Karlheinz Stockhausen** - experimented with texture-based composition\n- **Witold Lutosławski** - incorporated elements of sound mass\n\n## What is Sound Mass?\n\nSound mass (or *sonorité*) refers to treating sound itself—rather than traditional melody and harmony—as the primary compositional element. Musicians create dense, shimmering textures through:\n- Clusters of pitches\n- Microtonal shifts\n- Complex orchestral textures\n- Focus on timbre and density\n\nThis technique became particularly important in **spectral music** and modern classi

### Rewrite Claude 4.5 Haiku answers in the new column `gpt5mini_claude`:

In [5]:
# Look for where the generation was left off

last_row_id = -1

if os.path.exists(output_file):
  output_df = pd.read_json(output_file, lines = True) # JSONL
  last_row_id = output_df.index[-1]

print('No output file exists.' if last_row_id < 0 else f'Last row generated (row id): {last_row_id}')

Last row generated (row id): 5305


In [6]:
# Start appending to the file from where it's left off

with open(output_file, 'a') as f:

  # Keep track no-answer rows
  no_answer = 0

  # Iterate through each record/row
  for i, row in df.iterrows():

    # Print that the generation is already done if no rows are left to be generated
    if i == len(df) - 1:
      print(f'GPT5-mini {len(df)} answer generations already completed!')

    # Continue if current row is less than or equal to last_row_id
    if i <= last_row_id:
      continue

    # Convert each row to dictionary becasue the output file is going to be json
    row = row.to_dict()

    # Get the original answer to be rewritten
    original = row['claude_haiku_4.5']

    # Generate response from GPT-5mini
    try:
      gpt5mini_response = client.chat.completions.create(model = 'gpt-5-mini',
                                                         messages = [{'role': 'user', 'content': f"Refine the following text:\n\n{original}\n\nOutput only the rewritten text, nothing else."}],
                                                         max_completion_tokens = 1024,
                                                         reasoning_effort = 'low')  # Increase efficiency and comparability between models
      gpt5mini_answer = gpt5mini_response.choices[0].message.content

    except Exception as e:
      print(f'Error when processing row {i + 1} (index + 1) answer: {e}')
      no_answer += 1
      gpt5mini_answer = None

    # Insert the GPT-5mini answers into the dictionary (each row has a new column in the output file)
    row['gpt5mini_claude'] = gpt5mini_answer

    # Write the row with new column into the output file
    f.write(json.dumps(row) + '\n')   # '\n' so that each row starts with a new line


    # Print out the progress when generating
    if i % 100 == 0:

      # Force writing to the os in case the program crashes
      f.flush()
      os.fsync(f.fileno())

      # Print the progress and saving status
      if i != 0 and (i - last_row_id) > 100:
        print('Finished and saved!')
      print(f'Start generating 100 answers for batch {(i // 100) + 1} ....', end = ' ')

    elif i == len(df) - 1:
      print(f'Finished {len(df)} answer generations!')

GPT5-mini 5306 answer generations already completed!


In [7]:
# Convert JSONL to CSV
out_df = pd.read_json(output_file, lines = True)

output_file_csv = os.path.join(project_directory, 'data/gpt data/gpt5mini_claude.csv')
out_df.to_csv(output_file_csv, index = False)

### Read the output file after rewriting:

In [8]:
out_df = pd.read_csv(output_file_csv)
print('Sucessfully read the output file with GPT5-mini rewriting answers!')

print(f'The output file contains {len(out_df)} records/rows.')
print(f'New column names: {list(out_df.columns)}')

print('\nThe first 5 rows:')
display(out_df.head())

Sucessfully read the output file with GPT5-mini rewriting answers!
The output file contains 5306 records/rows.
New column names: ['question', 'human_answers', 'source', 'gemini_answers', 'gpt5mini_answer', 'claude_haiku_4.5', 'kimi_k2_answers', 'gpt5mini_claude']

The first 5 rows:


,question,human_answers,source,gemini_answers,gpt5mini_answer,claude_haiku_4.5,kimi_k2_answers,gpt5mini_claude
0,what composer used sound mass,"['Composers and works include Barbara Kolb , P...",open_qa,The most prominent composer strongly associate...,"Several 20th-century composers used the ""sound...",# Composers Who Used Sound Mass\n\n**Gyorgy Li...,"The term **""sound mass""** (or **""sound-mass co...",Composers Who Used Sound Mass\n\nGyörgy Ligeti...
1,where did the persian war take place,['The Greco-Persian Wars (also often called th...,open_qa,The Persian Wars primarily took place in **mai...,If you mean the Greco–Persian Wars (often call...,# The Persian Wars\n\nThe Persian Wars took pl...,The **Persian Wars** (also called the **Greco-...,The Persian Wars (5th century BCE) were fought...
2,what are add ons,"['Plug-in (computing) , a piece of software wh...",open_qa,An **add-on** is something extra or additional...,"""Add-ons"" (sometimes written ""add ons"", ""addon...",# Add-ons\n\nAdd-ons are **additional features...,**Add-ons** are extra software components or e...,"Add-ons\n\nAdd-ons are additional features, pr..."
3,how does a dredge work?,['Dredging is an excavation activity or operat...,open_qa,A dredge is essentially a floating machine des...,"A dredge is a machine used to remove sediment,...",# How a Dredge Works\n\nA dredge is a vessel o...,A **dredge** is a machine or system designed t...,How a Dredge Works\n\nA dredge is a vessel or ...
4,what classes are considered humanities,['The humanities are academic disciplines that...,open_qa,The humanities are academic disciplines that s...,Humanities are academic disciplines that study...,# Humanities Classes\n\nHumanities typically i...,“Humanities” is the broad family of discipline...,Humanities Classes\n\nHumanities encompass cou...


### Print the sample question and Claude 4.5 Haiku answers:

In [9]:
print('Sample question:')
display(out_df['question'].iloc[-1])

print('\nSample Claude original answer:')
display(out_df['claude_haiku_4.5'].iloc[-1])
print('\nSample Claude answer rewritten by GPT5-mini:')
display(out_df['gpt5mini_claude'].iloc[-1])

Sample question:


'Are BP of 119/65 and pulse of 35 causes for concern?Yesterday I was very dizzy, short of breath, heart rate around 35,BP 119/65. I am 51,female, overweight and have RA and Fibermyalgia. Do you know what might have caused this episode yesterday? I figure I would not feel okay today if it had been a heart attack.'


Sample Claude original answer:


'# Your Symptoms Need Medical Evaluation\n\nWhile I understand your reasoning, **I can\'t reassure you that this wasn\'t serious, and you should see a doctor promptly** — ideally today or tomorrow. Here\'s why:\n\n## The Concerning Part\nA heart rate of 35 (severe bradycardia) combined with dizziness and shortness of breath is a **red flag combination**, even if you feel better now. This matters because:\n\n- Some cardiac events don\'t always feel dramatic in the moment\n- Symptoms can resolve while underlying problems remain\n- The combo of low heart rate + SOB can indicate rhythm problems, medication effects, or other conditions needing diagnosis\n\n## What Could Cause This?\nWithout examining you, possibilities include:\n- Medication side effects (some RA meds affect heart rate)\n- Heart rhythm issues (not always a "heart attack")\n- Thyroid problems\n- Dehydration or medication interaction\n- Vasovagal episode\n- Other conditions\n\n## What to Do\n**Schedule an urgent appointment**


Sample Claude answer rewritten by GPT5-mini:


'Your symptoms should be evaluated by a clinician promptly — ideally today or tomorrow. I can’t reassure you this wasn’t serious.\n\nWhy this is concerning\n- A heart rate of 35 bpm (severe bradycardia) together with dizziness and shortness of breath is a red flag, even if you feel better now.\n- Some cardiac problems or rhythm disturbances can cause transient symptoms that resolve while an underlying issue persists.\n- Low heart rate plus shortness of breath can reflect rhythm problems, medication effects, thyroid dysfunction, dehydration, or other conditions that require diagnosis.\n\nPossible causes (cannot be confirmed without exam/tests)\n- Medication side effects or interactions (some RA medications and others can affect heart rate)\n- Heart rhythm disorders\n- Thyroid disease\n- Dehydration or electrolyte imbalance\n- Vasovagal episode\n- Other cardiac or noncardiac conditions\n\nWhat to do now\n- Arrange an urgent medical appointment (not routine). Tell them you had a heart rat

### Check NaN counts

In [10]:
pd.isna(out_df['gpt5mini_claude']).value_counts()

,count
gpt5mini_claude,
False,5296
True,10
